# navigation

> The navbar

In [1]:
#| default_exp navigation

A navbar names the application and offers its main routes. `NavBar` combines a brand link, a `nav` of links, and actions such as search or an account menu.

In [2]:
from fasthtml.common import Header, Nav, A, Div, Button, Input, Hr, unqid
from fastlucide.icons import Menu
from fastcore.utils import *
from answerui.registry import hypelt, reg
from answerui.layout import Cluster
from answerui.surfaces import Avatar
from answerui.overlays import Dropdown
from answerui.demo import example
import answerui.base  # the reset and tokens this styling is written on
import answerui.controls  # the controls the example places in the bar

In [3]:
from fasthtml.common import H1, P
from answerui.registry import reg
from answerui.layout import Stack
from answerui.surfaces import Badge
from answerui.demo import demo

## Navbar

The bar sits on the page colour, with a hairline beneath it, and the brand, the links, and the actions lie along it. Links are plain text; the current page carries `aria-current="page"` and takes the accent wash, as a chosen button does. `align` moves the links to the centre or the end of the space after the brand, `sticky` keeps the bar at the top as the page scrolls, and tells the document how tall it is, so a link to an anchor scrolls the anchor to just below the bar rather than under it; `vivid` and `dark` recolour it. The bar is marked `anchor`, so a popover opened from it, such as an account menu, sits below the bar rather than below its button.

When the bar is narrower than `--collapse-below`, or its content no longer fits on one line, the links fold away behind a disclosure button. The button is `quiet`, so it has no face or border until it is hovered or open. Its `aria-expanded` is the state, and CSS reads it to show the content in flow beneath the bar, so nothing is positioned and nothing is duplicated. Following a link closes it again, and so does a click outside the bar or Escape, which also returns focus to the button. Opening a menu from the bar closes it too, so the menu hangs below the bar's one row rather than below the links. Content passed as `end` stays in the bar, so an account menu is always to hand:

In [4]:
_navbar_css = r"""
[navbar] {
  display: flex; flex-wrap: wrap; align-items: center; gap: var(--space-2) var(--space);
  padding: var(--space-2) var(--space);
  --face: var(--n-3); background: var(--face); color: var(--n2);
  border-block-end: 1px solid var(--n-1);
  > a:first-child {
    display: flex; align-items: center; gap: var(--space-3); flex-shrink: 0; white-space: nowrap;
    font-weight: var(--weight1); font-size: var(--text1); color: inherit; text-decoration: none;
  }
  > [bar-content] { display: contents; }
  nav { display: flex; flex-wrap: wrap; gap: var(--space-3); }
  nav a { display: inline-block; padding: var(--space-3) var(--space-2); border-radius: var(--radius-1); color: inherit; text-decoration: none; }
  nav a:is(:hover, :focus-visible) { background: var(--tint); }
  nav a[aria-current=page] { background: var(--v-3); color: var(--v2); }
  > [end] { margin-inline-start: auto; --gap: var(--space-2); }
  > [burger] { display: none; padding: var(--space-3); }
}
"""

@hypelt(sel='[navbar]', layer='components', css=_navbar_css)
def NavBar(
    brand,          # Brand content, linked to href
    *c,             # Navigation and other content that collapses with it
    end=(),         # One element or a sequence that stays visible on narrow screens
    href='/',       # Where the brand links
    collapse=False, # Fold c behind a disclosure on a narrow bar, or when it no longer fits
    **attrs,        # Attributes for the header
):
    "The bar at the top of a page: brand, links, and actions"
    id = unqid()
    content = Div(*c, bar_content=True, id=id) if c else None
    burger = Button(Menu(aria_hidden='true'), type='button', quiet=True, burger=True, aria_controls=id, aria_expanded='false', aria_label='Navigation') if collapse and c else None
    return Header(A(brand, href=href), content, Cluster(*listify(end), end=True), burger,
        navbar=True, anchor=True, collapse=collapse, **attrs)

NavBar.variant('align', r"""
&[align=center] { nav { margin-inline: auto; } > [end] { margin-inline-start: 0; } }
&[align=end] { nav { margin-inline-start: auto; } > [end] { margin-inline-start: 0; } }
""", layer='components', doc='Where the links sit in the space after the brand: `center`, or `end` beside the actions.')
NavBar.variant('sticky', 'position: sticky; inset-block-start: 0; z-index: 1;', layer='components', doc='Stays at the top as the page scrolls, and keeps a scrolled-to anchor from landing under it.',
    on={('&', 'init'): r'''const set = () => document.documentElement.style.setProperty('--navbar-size', el.getBoundingClientRect().height + 'px');
set(); new ResizeObserver(set).observe(el)'''})
reg.rule('anchors', 'html { scroll-padding-block-start: calc(var(--navbar-size, 0px) + var(--space)); }', layer='components', doc='A scrolled-to anchor stops below the sticky navbar, with a little space.')
NavBar.variant('vivid', r"""
--face: var(--v); color: var(--n-3);
nav a[aria-current=page] { background: oklch(from var(--tint) l c h / calc(alpha * 2)); color: inherit; }
""", layer='components', doc='The brand and links on the accent.')
NavBar.variant('dark', 'color-scheme: dark;', layer='components', doc='A dark bar whatever the page scheme.')

_collapse_on = {
    ('& > [burger]', 'click'): "el.setAttribute('aria-expanded', el.getAttribute('aria-expanded') !== 'true')",
    ('& > [bar-content] a[href]', 'click'): "el.closest('[navbar]').querySelector(':scope > [burger]')?.setAttribute('aria-expanded', 'false')",
    ('& [popover]', 'beforetoggle', True): "if (e.newState === 'open') el.closest('[navbar]').querySelector(':scope > [burger]')?.setAttribute('aria-expanded', 'false')",
    (':root', 'click'): r"for (const b of document.querySelectorAll('[navbar] > [burger][aria-expanded=true]')) if (!b.closest('[navbar]').contains(e.target)) b.setAttribute('aria-expanded', 'false')",
    (':root', 'keydown'): r"""
if (e.key !== 'Escape') return;
for (const b of document.querySelectorAll('[navbar] > [burger][aria-expanded=true]')) { b.setAttribute('aria-expanded', 'false'); b.focus(); }""",
    ('&', 'init'): r"""
if (el._collapse) return;
const rem = parseFloat(getComputedStyle(document.documentElement).fontSize);
el._collapse = new ResizeObserver(() => {
  el.removeAttribute('collapsed');
  const below = parseFloat(getComputedStyle(el).getPropertyValue('--collapse-below')) * rem;
  el.toggleAttribute('collapsed', el.scrollWidth > el.clientWidth || el.clientWidth < below);
});
el._collapse.observe(el);"""}

NavBar.variant('collapse', r"""
--collapse-below: 48rem; flex-wrap: nowrap;
> [bar-content] > *, > [end] { flex-shrink: 0; }
nav { flex-wrap: nowrap; }
&[collapsed] {
  flex-wrap: wrap;
  > [burger] { display: inline-flex; }
  > [bar-content] { display: none; order: 1; flex-basis: 100%; }
  &:has(> [burger][aria-expanded=true]) > [bar-content] { display: grid; gap: var(--space-2); }
  nav { display: grid; margin: 0; }
}
""", on=_collapse_on, layer='components', doc='Folds the content behind a disclosure when the bar is narrower than `--collapse-below` or its content no longer fits; `collapsed` is the state its observer sets. A click outside the bar or Escape closes it.')

NavBar.collapse [components] :is([navbar])[collapse]: Folds the content behind a disclosure when the bar is narrower than `--collapse-below` or its content no longer fits; `collapsed` is the state its observer sets. A click outside the bar or Escape closes it.

A dashboard bar: links beside the brand, a search field that collapses with them, and an account menu that does not. The menu's trigger is a button named Account that holds only an avatar, which makes it round:

In [5]:
def _links(): return Nav(A('Overview', href='#', aria_current='page'), A('Inventory', href='#'), A('Customers', href='#'), A('Products', href='#'))

@example(NavBar)
def _navbar_example():
    return NavBar('Acme', _links(), Div(Input(type='search', placeholder='Search…', aria_label='Search'), role='search'),
        end=Dropdown(Button(Avatar('Ada Lovelace', style='--avatar-size: 1.75rem'), aria_label='Account'), A('Profile', href='#'), A('Settings', href='#'), Hr(), Button('Sign out', type='button')),
        collapse=True)

In [6]:
NavBar.demo(height='14rem')

<header navbar anchor collapse>
<a href="/">Acme</a>  <div bar-content id="_luSaN60-SiC5AaN5cTcEJA">
    <nav>
<a href="#" aria-current="page">Overview</a><a href="#">Inventory</a><a href="#">Customers</a><a href="#">Products</a>    </nav>
    <div role="search">
      <input type="search" placeholder="Search…" aria-label="Search">
    </div>
  </div>
  <div cluster end>
<button aria-label="Account" type="button" popovertarget="_ABX1IAUvTkyH_iIL1XMKzQ"><span role="img" aria-label="Ada Lovelace" avatar style="--avatar-size: 1.75rem">AL</span></button>    <div popover="auto" anchored dropdown id="_ABX1IAUvTkyH_iIL1XMKzQ">
<a href="#" autofocus>Profile</a><a href="#">Settings</a><hr><button type="button" popovertarget="_ABX1IAUvTkyH_iIL1XMKzQ" popovertargetaction="hide">Sign out</button>    </div>
  </div>
<button type="button" quiet burger aria-controls="_luSaN60-SiC5AaN5cTcEJA" aria-expanded="false" aria-label="Navigation"><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" aria-hidden="true" class="lucide-icon "><use href="#lc-menu"></use></svg></button></header>

## Configurations

The variants combine. A marketing site centres its links, or gathers them beside its sign-in buttons at the end. A vivid bar puts the brand on the accent, and a dark one stays dark on a light page:

In [7]:
def _auth(): return [Button('Log in', type='button', secondary=True), Button('Sign up', type='button')]
demo(Stack(
    NavBar('Acme', _links(), align='center'),
    NavBar('Acme', _links(), end=_auth(), align='end'),
    NavBar('Acme', _links(), end=_auth(), vivid=True),
    NavBar('Acme', _links(), end=_auth(), dark=True)))

<div stack>
  <header navbar anchor align="center">
<a href="/">Acme</a>    <div bar-content id="_WJZyP6pmTNOEpw_SW17pVg">
      <nav>
<a href="#" aria-current="page">Overview</a><a href="#">Inventory</a><a href="#">Customers</a><a href="#">Products</a>      </nav>
    </div>
    <div cluster end></div>
  </header>
  <header navbar anchor align="end">
<a href="/">Acme</a>    <div bar-content id="_xBQCJAq0SQqvtvxJrJnPaw">
      <nav>
<a href="#" aria-current="page">Overview</a><a href="#">Inventory</a><a href="#">Customers</a><a href="#">Products</a>      </nav>
    </div>
    <div cluster end>
<button type="button" secondary>Log in</button><button type="button">Sign up</button>    </div>
  </header>
  <header navbar anchor vivid>
<a href="/">Acme</a>    <div bar-content id="_PXCeNlp1SVS6iGhMpRu6Nw">
      <nav>
<a href="#" aria-current="page">Overview</a><a href="#">Inventory</a><a href="#">Customers</a><a href="#">Products</a>      </nav>
    </div>
    <div cluster end>
<button type="button" secondary>Log in</button><button type="button">Sign up</button>    </div>
  </header>
  <header navbar anchor dark>
<a href="/">Acme</a>    <div bar-content id="_nkaSlM6zRJ_09dQg2raChQ">
      <nav>
<a href="#" aria-current="page">Overview</a><a href="#">Inventory</a><a href="#">Customers</a><a href="#">Products</a>      </nav>
    </div>
    <div cluster end>
<button type="button" secondary>Log in</button><button type="button">Sign up</button>    </div>
  </header>
</div>

## Collapse

The preview is 24rem wide, so the bar collapses as it would on a phone, and the account menu stays inside the frame as it would stay inside the screen. Open the navigation and tab through it; the links expand below the bar while the account menu stays available.

In [8]:
demo(_navbar_example(), width='24rem', height='18rem')

<header navbar anchor collapse>
<a href="/">Acme</a>  <div bar-content id="__nywz3GMT8OsYHvK1QvPdg">
    <nav>
<a href="#" aria-current="page">Overview</a><a href="#">Inventory</a><a href="#">Customers</a><a href="#">Products</a>    </nav>
    <div role="search">
      <input type="search" placeholder="Search…" aria-label="Search">
    </div>
  </div>
  <div cluster end>
<button aria-label="Account" type="button" popovertarget="_XadCiSSiTau6Nc0cUPyajw"><span role="img" aria-label="Ada Lovelace" avatar style="--avatar-size: 1.75rem">AL</span></button>    <div popover="auto" anchored dropdown id="_XadCiSSiTau6Nc0cUPyajw">
<a href="#" autofocus>Profile</a><a href="#">Settings</a><hr><button type="button" popovertarget="_XadCiSSiTau6Nc0cUPyajw" popovertargetaction="hide">Sign out</button>    </div>
  </div>
<button type="button" quiet burger aria-controls="__nywz3GMT8OsYHvK1QvPdg" aria-expanded="false" aria-label="Navigation"><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" aria-hidden="true" class="lucide-icon "><use href="#lc-menu"></use></svg></button></header>

## Customisation

The bar owns its face, and an app owns the rest through a variant of its own. Here `glass` makes the face translucent over a blur of what scrolls beneath, sets the bar to the width of the content column, and lowers the collapse threshold to suit it. Scroll the preview to see the content through the bar:

In [9]:
reg.rule('column', ':root { --column: 42rem; }', doc='The width of the content column.')

NavBar.variant('glass', r"""
--face: oklch(from var(--n-2) l c h / .5); backdrop-filter: blur(16px) saturate(1.5);
max-inline-size: var(--column); margin: var(--space) auto 0; border: 1px solid var(--n-1); border-radius: var(--radius);
box-shadow: var(--floating);
--collapse-below: 30rem;
""", doc='A translucent bar at the width of the content column.')

demo(Div(
    NavBar('Acme', Nav(A('About', href='#', aria_current='page'), A('Blog', href='#'), A('Contact', href='#')),
        end=Button('Subscribe', type='button', secondary=True), sticky=True, glass=True, collapse=True),
    Stack(H1('Field notes'), Cluster(Badge('Design'), Badge('CSS', success=True), Badge('FastHTML', warning=True), gap='var(--space-2)'),
        P('A short account of building a component library on the web platform, written as it happened.'),
        P('The first decision was to style native tags rather than invent new ones. Every form control the browser already has is already accessible, already keyboard-driven, and already known to assistive technology.'),
        P('The second was that colour should come from one hue. A single number sets the whole palette, and severity is a context rather than a variant.'),
        style='max-inline-size: var(--column); margin-inline: auto; padding: var(--space1) var(--space)')),
    height='14rem')


<div>
  <header navbar anchor collapse sticky glass>
<a href="/">Acme</a>    <div bar-content id="_dgZCgbKwTUivUtadSALbHg">
      <nav>
<a href="#" aria-current="page">About</a><a href="#">Blog</a><a href="#">Contact</a>      </nav>
    </div>
    <div cluster end>
<button type="button" secondary>Subscribe</button>    </div>
<button type="button" quiet burger aria-controls="_dgZCgbKwTUivUtadSALbHg" aria-expanded="false" aria-label="Navigation"><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 24 24" height="24px" width="24px" aria-hidden="true" class="lucide-icon "><use href="#lc-menu"></use></svg></button>  </header>
  <div stack style="max-inline-size: var(--column); margin-inline: auto; padding: var(--space1) var(--space)">
    <h1>Field notes</h1>
    <div cluster style="--gap: var(--space-2);">
<span badge>Design</span><span badge success>CSS</span><span badge warning>FastHTML</span>    </div>
    <p>A short account of building a component library on the web platform, written as it happened.</p>
    <p>The first decision was to style native tags rather than invent new ones. Every form control the browser already has is already accessible, already keyboard-driven, and already known to assistive technology.</p>
    <p>The second was that colour should come from one hue. A single number sets the whole palette, and severity is a context rather than a variant.</p>
  </div>
</div>

## Export

In [10]:
#| hide
import nbdev; nbdev.nbdev_export()